In [23]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, classification_report, confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [24]:
# print(torch.cuda.is_available())
# print(torch.cuda.get_device_name(0))
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU Name: {torch.cuda.get_device_name(0)}")
print(f"PyTorch CUDA Version: {torch.version.cuda}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CUDA Available: True
GPU Name: NVIDIA GeForce RTX 4050 Laptop GPU
PyTorch CUDA Version: 12.6


In [25]:
df = pd.read_csv("fmnist_small.csv")

In [26]:
X = df.iloc[:, 1:]
Y = df.iloc[:, 0]

In [73]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [74]:
# Normalize image pixel (values in range of 0 to 1)
X_train /= 255.0
X_test /= 255.0

In [75]:
class CustomDataset(nn.Module):
    def __init__(self, input_feat, output_feat):
        self.input_feat = torch.tensor(input_feat, dtype=torch.float32).reshape(-1, 1, 28,28)
        self.output_feat = torch.tensor(output_feat, dtype=torch.long)

    def __len__(self):
        return len(self.input_feat)
    
    def __getitem__(self, index):
        return self.input_feat[index], self.output_feat[index]

In [76]:
train_dataset = CustomDataset(X_train.values, Y_train.values)
test_dataset = CustomDataset(X_test.values, Y_test.values)

In [77]:
train_loader = DataLoader(train_dataset, batch_size=32, drop_last=True, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, drop_last=True, shuffle=False)

### NN CLass

In [78]:
class myCNN(nn.Module):
    def __init__(self, input_features):
        super().__init__()

        # Feature Extractor -> it recognizes the patters from the images. 
        self.features = nn.Sequential(
            nn.Conv2d(input_features, 32, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, 64, kernel_size=3, padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Feature Classifier -> makes the final decision
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7, 128),
            nn.ReLU(),
            nn.Dropout(p=0.4),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(p=0.4),

            nn.Linear(64, 10)
        )

    def forward(self, X):
        X = self.features(X)
        X = self.classifier(X)
        return(X)



In [79]:
learning_rate = 0.001


epochs = 50

In [84]:
model = myCNN(1).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)

In [85]:
def evaluate_model(model, test_loader, criterion):
    """
    Evaluate the model on test data
    """
    model.eval()  # Set model to evaluation mode
    total_loss = 0
    all_predictions = []
    all_targets = []
    
    with torch.no_grad():  # Disable gradient computation
        for input_feat, output_feat in test_loader:
            # Move Data to GPU
            input_feat = input_feat.to(device)
            output_feat = output_feat.to(device)
            
            outputs = model(input_feat)
            loss = criterion(outputs, output_feat)
            total_loss += loss.item()
            
            # Get predictions
            _, predicted = torch.max(outputs, 1)

            all_predictions.extend(predicted.cpu().numpy())
            all_targets.extend(output_feat.cpu().numpy())
    
    avg_loss = total_loss / len(test_loader)
    return avg_loss, all_predictions, all_targets

def calculate_accuracy(predictions, targets):
    predictions = np.array(predictions)
    targets = np.array(targets)
    accuracy = np.mean(predictions == targets) * 100
    return accuracy

def predict_single_image(model, image):
    model.eval()
    with torch.no_grad():
        if len(image.shape) == 3:
            image = image.unsqueeze(0)
        image = image.to(device)
        
        outputs = model(image)
        _, predicted = torch.max(outputs, 1)
        probabilities = torch.softmax(outputs, dim=1)
        
    return predicted.item(), probabilities.cpu().numpy()

def predict_batch(model, images):
    model.eval()
    with torch.no_grad():
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        probabilities = torch.softmax(outputs, dim=1)
        
    return predicted.cpu().numpy(), probabilities.cpu().numpy()

In [86]:
train_losses = []
test_losses = []
test_accuracies = []

for epoch in range(epochs):
    model.train()  # Set to training mode
    total_epochs_loss = 0
    
    for input_feat, output_feat in train_loader:
        
        # Move Data to GPU
        input_feat = input_feat.to(device)
        output_feat = output_feat.to(device)
        
        outputs = model(input_feat)
        loss = criterion(outputs, output_feat)
        
        optimizer.zero_grad()  # Note the parentheses!
        loss.backward()
        optimizer.step()
        
        total_epochs_loss += loss.item()
    
    avg_train_loss = total_epochs_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    # Evaluate on test set
    test_loss, predictions, targets = evaluate_model(model, test_loader, criterion)
    accuracy = calculate_accuracy(predictions, targets)
    test_losses.append(test_loss)
    test_accuracies.append(accuracy)
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Train Loss: {avg_train_loss:.4f}, Test Loss: {test_loss:.4f}, Test Accuracy: {accuracy:.2f}%')

Epoch [10/50], Train Loss: 0.1744, Test Loss: 0.4430, Test Accuracy: 85.90%
Epoch [20/50], Train Loss: 0.0679, Test Loss: 0.7197, Test Accuracy: 84.88%
Epoch [30/50], Train Loss: 0.0534, Test Loss: 0.8538, Test Accuracy: 86.32%
Epoch [40/50], Train Loss: 0.0577, Test Loss: 1.0010, Test Accuracy: 86.06%
Epoch [50/50], Train Loss: 0.0552, Test Loss: 1.0654, Test Accuracy: 85.47%


In [87]:
print("FINAL EVALUATION ON TEST SET")
print("="*50)

# Run final evaluation - returns (avg_loss, predictions, targets)
test_loss, predictions, targets = evaluate_model(model, test_loader, criterion)
final_accuracy = calculate_accuracy(predictions, targets)

print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {final_accuracy:.2f}%")

print("\nClassification Report:")
print(classification_report(targets, predictions, digits=3))

print("\nConfusion Matrix:")
print(confusion_matrix(targets, predictions))


FINAL EVALUATION ON TEST SET

Test Loss: 1.0654
Test Accuracy: 85.47%

Classification Report:
              precision    recall  f1-score   support

           0      0.782     0.776     0.779       143
           1      0.991     0.958     0.975       120
           2      0.764     0.822     0.792       118
           3      0.822     0.888     0.854       125
           4      0.812     0.704     0.754       135
           5      0.990     0.920     0.954       112
           6      0.651     0.661     0.656       127
           7      0.934     0.925     0.930       107
           8      0.982     1.000     0.991       110
           9      0.906     1.000     0.951        87

    accuracy                          0.855      1184
   macro avg      0.863     0.865     0.863      1184
weighted avg      0.856     0.855     0.854      1184


Confusion Matrix:
[[111   1   3   6   2   0  20   0   0   0]
 [  0 115   1   3   0   0   1   0   0   0]
 [  5   0  97   1  11   0   4   0   0   0]